# IM2ELEVATION DSM Sample Visualizer

This notebook visualizes DSM test samples including RGB, ground truth DSM, predicted DSM, and their difference.
The DSMs are filtered using semantic segmentation masks to show only building regions.
Uses predictions from the IM2ELEVATION model pipeline output.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from pathlib import Path
import random
from scipy import ndimage  # For resizing predicted DSM if needed
import glob

dataset_name = 'DFC2023S'  # Change this to the desired dataset name, like 'DFC2023S', 'DFC2019_crp512_bin', or 'Huawei_Contest'

# Define dataset paths based on the dataset name
test_dataset_path = f"/home/asfand/Ahmad/datasets/{dataset_name}/test"
pred_output_path = f"/home/asfand/Ahmad/IM2ELEVATION/pipeline_output/{dataset_name}/predictions"

# Sample selection options
use_random_sample = True  # Set to True for random selection, False for specific sample
sample_name = "OMA_163_037_crop02"  # Specific sample name (used when use_random_sample=False)

# Helper function to detect file extensions in a directory
def detect_file_extensions(directory):
    """Detect the file extensions used in a directory."""
    if not os.path.exists(directory):
        return []
    
    extensions = set()
    for file in os.listdir(directory):
        if os.path.isfile(os.path.join(directory, file)):
            _, ext = os.path.splitext(file)
            if ext:
                extensions.add(ext.lower())
    
    return list(extensions)

# Helper function to find file with any supported extension
def find_file_with_extension(base_path, filename_without_ext, supported_extensions=None):
    """Find a file with the given name and any of the supported extensions."""
    if supported_extensions is None:
        supported_extensions = ['.tif', '.tiff', '.png', '.jpg', '.jpeg', '.npy']
    
    for ext in supported_extensions:
        full_path = f"{base_path}/{filename_without_ext}{ext}"
        if os.path.exists(full_path):
            return full_path, ext
    
    return None, None

# Get all sample names from the dataset
def get_all_sample_names():
    """Get all sample names by scanning the RGB directory and extracting base names."""
    rgb_dir = os.path.join(test_dataset_path, "rgb")
    if not os.path.exists(rgb_dir):
        return []
    
    # Get all image files from RGB directory
    sample_names = []
    for file in os.listdir(rgb_dir):
        if os.path.isfile(os.path.join(rgb_dir, file)):
            # Remove extension to get base name
            base_name = os.path.splitext(file)[0]
            sample_names.append(base_name)
    
    return sample_names

# Function to check if all required files exist for a sample
def check_sample_files_exist(sample_name):
    """Check if all required files (RGB, DSM, SEM, Predicted DSM) exist for a sample."""
    rgb_dir = os.path.join(test_dataset_path, "rgb")
    dsm_dir = os.path.join(test_dataset_path, "dsm")
    sem_dir = os.path.join(test_dataset_path, "sem")
    
    missing_files = []
    
    # Check RGB file
    rgb_path, rgb_ext = find_file_with_extension(rgb_dir, sample_name)
    if rgb_path is None:
        missing_files.append("RGB")
    
    # Check GT DSM file
    dsm_path, dsm_ext = find_file_with_extension(dsm_dir, sample_name)
    if dsm_path is None:
        missing_files.append("GT DSM")
    
    # Check GT SEM file
    sem_path, sem_ext = find_file_with_extension(sem_dir, sample_name)
    if sem_path is None:
        missing_files.append("GT SEM")
    
    # Check Predicted DSM file - look for both .npy and other formats
    pred_dsm_path_npy, _ = find_file_with_extension(pred_output_path, f"{sample_name}_pred", ['.npy'])
    pred_dsm_path_regular, _ = find_file_with_extension(pred_output_path, sample_name, ['.tif', '.tiff', '.png'])
    
    if pred_dsm_path_npy is None and pred_dsm_path_regular is None:
        missing_files.append("Predicted DSM")
    
    return len(missing_files) == 0, missing_files

# Get sample name based on selection mode
def get_sample_name():
    if use_random_sample:
        # Get list of available samples that have all required files
        sample_names = get_all_sample_names()
        
        if sample_names:
            # Filter samples that have all required files
            valid_samples = []
            for sample in sample_names:
                all_exist, missing = check_sample_files_exist(sample)
                if all_exist:
                    valid_samples.append(sample)
            
            if valid_samples:
                selected_sample = random.choice(valid_samples)
                print(f"Randomly selected sample: {selected_sample}")
                print(f"Found {len(valid_samples)} samples with complete files out of {len(sample_names)} total samples")
                return selected_sample
            else:
                print(f"No samples found with all required files (RGB, DSM, SEM, Predicted DSM)")
                print(f"Total samples: {len(sample_names)}")
                # Show some examples of missing files
                for i, sample in enumerate(sample_names[:5]):
                    all_exist, missing = check_sample_files_exist(sample)
                    if not all_exist:
                        print(f"  Sample '{sample}' missing: {', '.join(missing)}")
                return None
        else:
            print(f"No samples found in RGB directory: {os.path.join(test_dataset_path, 'rgb')}")
            return None
    else:
        print(f"Using specified sample: {sample_name}")
        all_exist, missing = check_sample_files_exist(sample_name)
        if all_exist:
            return sample_name
        else:
            print(f"Sample '{sample_name}' missing files: {', '.join(missing)}")
            return None

# Get the actual sample name to use
current_sample_name = get_sample_name()

In [ ]:
def load_sample_data(sample_name):
    """Load RGB, GT DSM, GT SEM, and predicted DSM for a given sample."""
    
    # Directory paths
    rgb_dir = os.path.join(test_dataset_path, "rgb")
    dsm_dir = os.path.join(test_dataset_path, "dsm")
    sem_dir = os.path.join(test_dataset_path, "sem")
    
    # Find file paths with flexible extensions
    rgb_path, rgb_ext = find_file_with_extension(rgb_dir, sample_name)
    gt_dsm_path, dsm_ext = find_file_with_extension(dsm_dir, sample_name)
    gt_sem_path, sem_ext = find_file_with_extension(sem_dir, sample_name)
    
    # For predicted DSM, check for both _pred.npy format and regular formats
    pred_dsm_path_npy, _ = find_file_with_extension(pred_output_path, f"{sample_name}_pred", ['.npy'])
    pred_dsm_path_regular, pred_ext = find_file_with_extension(pred_output_path, sample_name, ['.tif', '.tiff', '.png'])
    
    # Prefer .npy format for IM2ELEVATION predictions
    if pred_dsm_path_npy:
        pred_dsm_path = pred_dsm_path_npy
        pred_ext = '.npy'
    else:
        pred_dsm_path = pred_dsm_path_regular
    
    # Load images
    try:
        rgb_img = np.array(Image.open(rgb_path))
        gt_dsm = np.array(Image.open(gt_dsm_path)).astype(np.float32)
        gt_sem = np.array(Image.open(gt_sem_path)).astype(np.uint8)
        
        # Load predicted DSM based on file format
        if pred_ext == '.npy':
            pred_dsm = np.load(pred_dsm_path).astype(np.float32)
        else:
            pred_dsm = np.array(Image.open(pred_dsm_path)).astype(np.float32)
        
        print(f"Successfully loaded sample: {sample_name}")
        print(f"RGB shape: {rgb_img.shape} (format: {rgb_ext})")
        print(f"GT DSM shape: {gt_dsm.shape}, range: [{gt_dsm.min():.2f}, {gt_dsm.max():.2f}] (format: {dsm_ext})")
        print(f"GT SEM shape: {gt_sem.shape}, unique values: {np.unique(gt_sem)} (format: {sem_ext})")
        print(f"Pred DSM shape: {pred_dsm.shape}, range: [{pred_dsm.min():.2f}, {pred_dsm.max():.2f}] (format: {pred_ext})")
        
        return rgb_img, gt_dsm, gt_sem, pred_dsm
        
    except Exception as e:
        print(f"Error loading sample {sample_name}: {e}")
        print(f"Attempted paths:")
        print(f"  RGB: {rgb_path}")
        print(f"  GT DSM: {gt_dsm_path}")
        print(f"  GT SEM: {gt_sem_path}")
        print(f"  Pred DSM: {pred_dsm_path}")
        return None, None, None, None

In [ ]:
def apply_semantic_masks(gt_dsm, pred_dsm, gt_sem):
    """Apply semantic segmentation masks to filter building regions only."""
    
    # Check if predicted DSM needs resizing to match ground truth dimensions
    if pred_dsm.shape != gt_dsm.shape:
        print(f"Resizing predicted DSM from {pred_dsm.shape} to {gt_dsm.shape}")
        from scipy import ndimage
        # Resize predicted DSM to match ground truth dimensions
        zoom_factors = (gt_dsm.shape[0] / pred_dsm.shape[0], gt_dsm.shape[1] / pred_dsm.shape[1])
        pred_dsm = ndimage.zoom(pred_dsm, zoom_factors, order=1)  # Linear interpolation
    
    # Create binary mask for buildings (class 1) from ground truth segmentation
    gt_mask = (gt_sem == 1).astype(np.float32)
    
    # Store original values
    gt_dsm_original = gt_dsm.copy()
    pred_dsm_original = pred_dsm.copy()
    
    # Apply ground truth mask to both GT and predicted DSMs
    gt_dsm_masked = gt_dsm * gt_mask
    pred_dsm_masked = pred_dsm * gt_mask
    
    # Calculate statistics
    building_pixels = np.sum(gt_mask)
    total_pixels = gt_mask.size
    building_percentage = 100.0 * building_pixels / total_pixels
    
    print(f"Building pixels: {building_pixels} / {total_pixels} ({building_percentage:.2f}%)")
    
    return gt_dsm_masked, pred_dsm_masked, gt_mask

In [ ]:
def visualize_dsm_analysis(rgb_img, gt_dsm, pred_dsm, gt_mask, sample_name):
    """Create a 4-panel visualization similar to the attached screenshot."""
    
    # Calculate DSM difference
    dsm_diff = gt_dsm - pred_dsm
    
    # Create figure with 4 subplots
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'IM2ELEVATION DSM Analysis: {sample_name}', fontsize=16, fontweight='bold')
    
    # Top-left: RGB Image
    axes[0, 0].imshow(rgb_img)
    axes[0, 0].set_title('RGB Image')
    axes[0, 0].axis('off')
    
    # Top-right: Ground Truth DSM
    gt_range = [gt_dsm.min(), gt_dsm.max()]
    im1 = axes[0, 1].imshow(gt_dsm, cmap='viridis', vmin=gt_range[0], vmax=gt_range[1])
    axes[0, 1].set_title(f'Ground Truth DSM\nRange: [{gt_range[0]:.1f}, {gt_range[1]:.1f}]')
    axes[0, 1].axis('off')
    cbar1 = plt.colorbar(im1, ax=axes[0, 1], shrink=0.8)
    cbar1.set_label('Height (m)')
    
    # Bottom-left: Predicted DSM
    pred_range = [pred_dsm.min(), pred_dsm.max()]
    im2 = axes[1, 0].imshow(pred_dsm, cmap='viridis', vmin=pred_range[0], vmax=pred_range[1])
    axes[1, 0].set_title(f'IM2ELEVATION Predicted DSM\nRange: [{pred_range[0]:.1f}, {pred_range[1]:.1f}]')
    axes[1, 0].axis('off')
    cbar2 = plt.colorbar(im2, ax=axes[1, 0], shrink=0.8)
    cbar2.set_label('Height (m)')
    
    # Bottom-right: DSM Difference (GT - Pred)
    diff_range = [dsm_diff.min(), dsm_diff.max()]
    im3 = axes[1, 1].imshow(dsm_diff, cmap='RdBu_r', vmin=diff_range[0], vmax=diff_range[1])
    axes[1, 1].set_title(f'DSM Difference (GT - Pred)\nRange: [{diff_range[0]:.1f}, {diff_range[1]:.1f}]')
    axes[1, 1].axis('off')
    cbar3 = plt.colorbar(im3, ax=axes[1, 1], shrink=0.8)
    cbar3.set_label('Height Difference (m)')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nDSM Statistics for {sample_name}:")
    print(f"GT DSM  - Min: {gt_dsm.min():.3f}, Max: {gt_dsm.max():.3f}, Mean: {gt_dsm.mean():.3f}")
    print(f"Pred DSM - Min: {pred_dsm.min():.3f}, Max: {pred_dsm.max():.3f}, Mean: {pred_dsm.mean():.3f}")
    print(f"Difference - Min: {dsm_diff.min():.3f}, Max: {dsm_diff.max():.3f}, Mean: {dsm_diff.mean():.3f}")
    
    # Calculate error metrics for building regions only
    building_mask = gt_mask > 0
    if np.any(building_mask):
        gt_buildings = gt_dsm[building_mask]
        pred_buildings = pred_dsm[building_mask]
        diff_buildings = dsm_diff[building_mask]
        
        mae = np.mean(np.abs(diff_buildings))
        rmse = np.sqrt(np.mean(diff_buildings**2))
        
        print(f"\nBuilding Region Metrics:")
        print(f"MAE: {mae:.3f} m")
        print(f"RMSE: {rmse:.3f} m")

In [ ]:
# Load the sample data
if current_sample_name is not None:
    rgb_img, gt_dsm, gt_sem, pred_dsm = load_sample_data(current_sample_name)

    if rgb_img is not None:
        # Apply semantic masks to filter building regions
        gt_dsm_masked, pred_dsm_masked, gt_mask = apply_semantic_masks(gt_dsm, pred_dsm, gt_sem)
        
        # Visualize the results
        visualize_dsm_analysis(rgb_img, gt_dsm_masked, pred_dsm_masked, gt_mask, current_sample_name)
    else:
        print(f"Failed to load sample: {current_sample_name}")
        print("\nAvailable samples in rgb directory:")
        rgb_dir = os.path.join(test_dataset_path, "rgb")
        if os.path.exists(rgb_dir):
            rgb_files = [f for f in os.listdir(rgb_dir) if f.endswith('.tif')]
            for i, f in enumerate(rgb_files[:10]):  # Show first 10 files
                print(f"  {i+1}. {f.replace('.tif', '')}")
            if len(rgb_files) > 10:
                print(f"  ... and {len(rgb_files) - 10} more files")
        else:
            print(f"RGB directory not found: {rgb_dir}")
else:
    print("No sample selected. Please check the dataset path and try again.")

## Sample Selection Options

You have two ways to select samples for visualization:

### Option 1: Specific Sample (Default)
- Set `use_random_sample = False` in the second cell
- Specify the desired `sample_name` (e.g., "OMA_163_037_crop02")
- Re-run the cells to visualize that specific sample

### Option 2: Random Sample
- Set `use_random_sample = True` in the second cell  
- Re-run the cells to randomly select and visualize a sample from the dataset
- Each run will select a different random sample

The notebook uses IM2ELEVATION model predictions and applies the same semantic segmentation masking logic:
- Buildings are encoded with value 1 in semantic segmentation
- Background regions are encoded with value 0
- Only building regions are shown in the DSM visualizations
- Error metrics are calculated only for building pixels
- Predictions are loaded from `/home/asfand/Ahmad/IM2ELEVATION/pipeline_output/{dataset_name}/predictions/`

## 🛠️ Complete Optional Normalization Pipeline

### New Feature: Full Normalization/Denormalization Control

I've made the **entire normalization pipeline** optional with a single consistent variable:

**Command Line Flag:**
```bash
--disable-normalization    # Disable entire normalization pipeline (default: False)
```

### Complete Pipeline Control:

**Default Behavior (Full Normalization Pipeline):**
1. **loaddata.py**: ×1000 (original DSM → scaled for processing) - **float32**
2. **nyu_transform.py**: ÷100,000 (normalize to [0-1] range for model) - **float32**
3. **test.py/util.py**: ×100 (restore to original DSM scale) - **float32**
4. **Net Effect**: ×1000 × (1/100,000) × 100 = **×1** (perfect restoration)

**With --disable-normalization:**
1. **loaddata.py**: No ×1000 scaling - **float32**
2. **nyu_transform.py**: No ÷100,000 normalization - **float32**
3. **test.py/util.py**: No ×100 restoration - **float32**
4. **Net Effect**: Raw model behavior with original data ranges

### Key Changes:
- **Consistent Variable**: Single `disable_normalization` flag throughout pipeline
- **Float32 Data Type**: Depth values maintained as float32 (not uint16) for higher precision
- **PIL Mode 'F'**: Enhanced ToTensor to handle float32 PIL images properly

### Usage Examples:

**1. Full Pipeline (Default - Recommended):**
```bash
python test.py --model pipeline_output/DFC2019_crp512_bin --csv dataset/test_DFC2019_crp512_bin.csv --save-predictions --batch-size 1 --gpu-ids 0
```

**2. Raw Model Analysis (No Normalization):**
```bash
python test.py --model pipeline_output/DFC2019_crp512_bin --csv dataset/test_DFC2019_crp512_bin.csv --save-predictions --batch-size 1 --gpu-ids 0 --disable-normalization
```

**3. Via Pipeline Script:**
```bash
./run_pipeline.sh --dataset DFC2019_crp512_bin --skip-csv --skip-training --disable-normalization
```

### Files Modified:
- **`test.py`**: Single `--disable-normalization` flag
- **`loaddata.py`**: Float32 data type, optional ×1000 scaling
- **`nyu_transform.py`**: Enhanced ToTensor for float32, optional ÷100,000 normalization
- **`util.py`**: Consistent variable naming, optional ×100 scaling
- **`run_pipeline.sh`**: Added `--disable-normalization` support

This provides complete, consistent control over the entire normalization pipeline with improved precision!